# 01_模型与消息

**来源:**
- https://docs.langchain.com/docs/models
- https://docs.langchain.com/docs/messages

## 1. 模型概述

Models（模型）是 Agent 的**推理引擎**，驱动 Agent 的决策过程——决定调用哪些工具、如何解释结果以及何时提供最终答案。

### 模型的能力

- **工具调用 (Tool Calling)**：调用外部工具并在响应中使用结果
- **结构化输出 (Structured Output)**：响应遵循定义的格式
- **多模态 (Multimodality)**：处理和返回文本以外的数据类型（图像、音频、视频）
- **推理 (Reasoning)**：执行多步推理以得出结论

### 支持的主要提供商

| 提供商 | 包名 | 模型格式 |
|--------|------|----------|
| OpenAI | `langchain-openai` | `openai:gpt-5.4` |
| Anthropic | `langchain-anthropic` | `claude-sonnet-4-6` |
| Google Gemini | `langchain-google-genai` | `google_genai:gemini-2.5-flash-lite` |
| AWS Bedrock | `langchain-aws` | `anthropic.claude-3-5-sonnet...` |
| Azure OpenAI | `langchain-openai` | `azure_openai:gpt-5.4` |
| HuggingFace | `langchain-huggingface` | `microsoft/Phi-3-mini-4k-instruct` |
| OpenRouter | `langchain-openrouter` | `openrouter:anthropic/claude-sonnet-4-6` |
| Fireworks | `langchain-fireworks` | `fireworks:accounts/fireworks/models/...` |
| Ollama | `langchain-ollama` | `ollama:devstral-2` |
| Baseten | `langchain-baseten` | `baseten:zai-org/GLM-5` |

## 2. 初始化模型

使用 `init_chat_model` 或直接使用提供商特定的类：

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
# 方法 1: 使用 init_chat_model（推荐）
from langchain.chat_models import init_chat_model

# OpenAI 模型
import os
os.environ["OPENAI_API_KEY"] = "sk-..."

model = init_chat_model("openai:gpt-5.4")

print(f"模型类型: {type(model).__name__}")

# 方法 2: 直接使用提供商类
from langchain_openai import ChatOpenAI

model2 = ChatOpenAI(model="gpt-5.4")
print(f"模型类型: {type(model2).__name__}")

In [ ]:
# 不同提供商的初始化示例

# Anthropic Claude
from langchain.chat_models import init_chat_model

# Gemini
model_gemini = init_chat_model("google_genai:gemini-2.5-flash-lite")
print(f"Gemini 模型: {model_gemini}")

# AWS Bedrock
model_bedrock = init_chat_model(
    "anthropic.claude-3-5-sonnet-20240620-v1:0",
    model_provider="bedrock_converse",
)
print(f"Bedrock 模型: {model_bedrock}")

# HuggingFace
model_hf = init_chat_model(
    "microsoft/Phi-3-mini-4k-instruct",
    model_provider="huggingface",
    temperature=0.7,
    max_tokens=1024,
)
print(f"HuggingFace 模型: {model_hf}")

## 3. 模型参数

| 参数 | 类型 | 默认值 | 描述 |
|------|------|--------|------|
| `model` | string | 必填 | 模型名称或标识符 |
| `temperature` | number | 提供商默认 | 控制输出的随机性（越高越有创意） |
| `max_tokens` | number | 提供商默认 | 限制响应中的最大 token 数 |
| `timeout` | number | 提供商默认 | 等待响应的最大时间（秒） |
| `max_retries` | number | 6 | 失败时的最大重试次数 |
| `api_key` | string | 环境变量 | API 密钥 |

In [ ]:
# 配置模型参数的完整示例
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "deepseek-v4-pro",
    temperature=0.7,       # 较高的随机性，更有创意
    timeout=30,            # 30秒超时
    max_tokens=1000,       # 最大输出 1000 tokens
    max_retries=6,         # 最多重试 6 次
)

print(f"参数配置完成: temp={model.temperature}, timeout={model.timeout}")

### 连接弹性

LangChain 的聊天模型自动使用指数退避策略重试失败的 API 请求：

- **自动重试**：网络错误、速率限制(429)、服务器错误(5xx)
- **不重试**：客户端错误如 401（未授权）或 404
- **默认重试次数**：6 次
- **建议**：在不可靠网络上的长时 Agent 任务中可增加到 10-15 次

## 4. 模型调用方法

### 4.1 Invoke（调用）

最直接的调用方式，生成完整的响应后返回：

In [ ]:
# 简单字符串输入
response = model.invoke("为什么鹦鹉有彩色的羽毛？")
print(f"响应: {response}")

# 消息列表输入（对话历史）
from langchain.messages import HumanMessage, AIMessage, SystemMessage

conversation = [
    SystemMessage("你是一个将英语翻译为法语的有用助手。"),
    HumanMessage("翻译: I love programming."),
    AIMessage("J'adore la programmation."),
    HumanMessage("翻译: I love building applications.")
]

response = model.invoke(conversation)
print(f"翻译结果: {response.content}")

### 4.2 Stream（流式输出）

实时生成输出，逐块返回：

In [ ]:
# 流式输出示例
for chunk in model.stream("为什么鹦鹉有彩色的羽毛？"):
    print(chunk.text, end="", flush=True)

print("\n\n--- 流式完成 ---")

### 4.3 批量处理

同时向模型发送多个请求，提高效率：

```python
# 批量处理示例
results = model.batch([
    "写一首关于春天的诗",
    "解释量子计算的基本原理",
    "什么是机器学习？"
])
```

## 5. 消息系统

Messages（消息）是 LangChain 中模型上下文的基本单元，代表与 LLM 交互时的输入和输出。

每条消息包含：
- **角色 (Role)**：识别消息类型（system, user, assistant, tool）
- **内容 (Content)**：实际内容（文本、图像、音频、文档等）
- **元数据 (Metadata)**：可选字段，如响应信息、消息 ID 和 token 用量

### 5.1 消息类型

| 类型 | 类名 | 用途 |
|------|------|------|
| **系统消息** | `SystemMessage` | 设置模型的初始行为、语气和指南 |
| **人类消息** | `HumanMessage` | 表示用户的输入和交互 |
| **AI 消息** | `AIMessage` | 模型生成的响应，包含文本和工具调用 |
| **工具消息** | `ToolMessage` | 工具执行的结果 |

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

model = init_chat_model("openai:gpt-5-nano")

# 系统消息：定义模型行为
system_msg = SystemMessage(
    "你是一个资深的 Python 开发者，擅长 Web 框架。"
    "总是提供代码示例并解释你的推理。"
    "解释要简洁但透彻。"
)

# 人类消息：用户输入
human_msg = HumanMessage("如何创建一个 REST API？")

# 组合消息并调用模型
messages = [system_msg, human_msg]
response = model.invoke(messages)
print(f"AI 响应类型: {type(response).__name__}")
print(f"响应内容: {response.content[:200]}...")

### 5.2 AI 消息属性

| 属性 | 类型 | 描述 |
|------|------|------|
| `text` | string | 消息的文本内容 |
| `content` | string \| dict[] | 原始消息内容 |
| `content_blocks` | ContentBlock[] | 标准化的内容块 |
| `tool_calls` | dict[] \| None | 模型发出的工具调用 |
| `id` | string | 消息的唯一标识符 |
| `usage_metadata` | dict \| None | token 用量元数据 |
| `response_metadata` | ResponseMetadata \| None | 提供商响应元数据 |

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("openai:gpt-5-nano")

# 调用模型并检查返回的 AI 消息
response = model.invoke("你好！")

print(f"文本: {response.text}")
print(f"Token 用量: {response.usage_metadata}")
print(f"工具调用: {response.tool_calls}")
print(f"响应元数据: {response.response_metadata}")

### 5.3 工具消息

当模型使用工具调用时，AI 消息包含 `tool_calls`。工具消息用于将单个工具执行的结果传回模型：

In [ ]:
from langchain.messages import AIMessage, ToolMessage, HumanMessage

# 模拟模型发出工具调用
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)

# 执行工具并创建结果消息
weather_result = "晴朗，72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123",  # 必须匹配工具调用 ID
    name="get_weather"
)

print(f"工具消息内容: {tool_message.content}")
print(f"工具调用 ID: {tool_message.tool_call_id}")

# `artifact` 字段：存储不会发送给模型的补充数据
from langchain.messages import ToolMessage

artifact_tool_msg = ToolMessage(
    content="It was the best of times...",
    tool_call_id="call_456",
    name="search_books",
    artifact={"document_id": "doc_123", "page": 0},
)
print(f"Artifact（不发送给模型）: {artifact_tool_msg.artifact}")

### 5.4 消息内容与多模态

消息的 `content` 属性支持字符串或内容块列表，包括多模态数据：

In [ ]:
from langchain.messages import HumanMessage

# 文本内容
msg1 = HumanMessage("你好！")

# 多模态内容（提供商原生格式）
msg2 = HumanMessage(content=[
    {"type": "text", "text": "描述这张图片的内容"},
    {"type": "image_url", "image_url": {"url": "https://example.com/image.jpg"}}
])

# 标准内容块格式
msg3 = HumanMessage(content_blocks=[
    {"type": "text", "text": "描述这张图片的内容"},
    {"type": "image", "url": "https://example.com/image.jpg"},
])

print(f"文本消息: {msg1.content}")
print(f"多模态消息: {msg2.content}")
print(f"标准内容块消息: {msg3.content_blocks}")

### 5.5 标准内容块类型

LangChain 提供跨提供商兼容的标准内容块表示：

| 类型 | 用途 | 必填字段 |
|------|------|----------|
| `text` | 标准文本输出 | `type`, `text` |
| `reasoning` | 模型推理步骤 | `type`, `reasoning` |
| `image` | 图像数据（URL/base64/file_id） | `type` + 其一 |
| `audio` | 音频数据 | `type` + 其一 |
| `video` | 视频数据 | `type` + 其一 |
| `file` | 文件数据 | `type` + 其一 |

In [ ]:
# 标准内容块示例
from langchain.messages import AIMessage

# 来自 Anthropic 的 thinking block
message_anthropic = AIMessage(
    content=[
        {"type": "thinking", "thinking": "...详细推理...", "signature": "WaUjzkyp..."},
        {"type": "text", "text": "最终答案..."},
    ],
    response_metadata={"model_provider": "anthropic"}
)

# 解析为标准内容块
blocks = message_anthropic.content_blocks
print(f"标准内容块: {blocks}")
# [{'type': 'reasoning', 'reasoning': '...', 'extras': {'signature': '...'}},
#  {'type': 'text', 'text': '...'}]

# 多模态输入示例
# 支持 URL、base64 编码、提供商管理的 File ID 三种方式
multimodal_msg = {
    "role": "user",
    "content": [
        {"type": "text", "text": "描述这个图像的内容"},
        {"type": "image", "url": "https://example.com/path/to/image.jpg"},
    ]
}
print(f"多模态输入: {multimodal_msg}")

In [ ]:
### 5.6 输出版本控制

如需标准内容块序列化到 `content` 字段（供 LangChain 外部应用使用），可设置环境变量：

```bash
export LC_OUTPUT_VERSION=v1
```

或在初始化模型时指定：

```python
model = init_chat_model("gpt-5-nano", output_version="v1")
```

---

**参考链接**
- https://docs.langchain.com/docs/models
- https://docs.langchain.com/docs/messages
- https://docs.langchain.com/docs/tools
- https://docs.langchain.com/docs/concepts/structured_output